<a href="https://colab.research.google.com/github/Moksh45/Running-Ollama-on-Google-Colab-Through-Pinggy/blob/main/Running-Ollama-on-Google-Colab-Through-Pinggy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
# Install pciutils and Ollama
!sudo apt-get install -y pciutils
!curl https://ollama.ai/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [2]:
# Install pinggy
!pip install pinggy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 77.1 MB/s eta 0:00:0000:0100:01


In [3]:
!nvidia-smi

Fri May  1 21:02:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
print("okay")

okay


In [4]:
# Start the Ollama server in a subprocess
import subprocess
def start_ollama_server():
    subprocess.Popen(['ollama', 'serve'])
    print("🚀 Ollama server launched successfully!")
start_ollama_server()

FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [17]:
# Check if Ollama is listening on port 11434
def check_ollama_port(port='11434'):
    try:
        output = subprocess.run(['sudo', 'lsof', '-i', '-P', '-n'],
                              capture_output=True, text=True).stdout
        if f":{port} (LISTEN)" in output:
            print(f"✅ Ollama is actively listening on port {port}")
        else:
            print(f"⚠️ Ollama not detected on port {port}")
    except Exception as e:
        print(f"❌ Error checking port: {e}")
check_ollama_port()

⚠️ Ollama not detected on port 11434


In [ ]:
# Start a pinggy tunnel to expose the Ollama API
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:11434",
    headermodification=["u:Host:localhost:11434"]
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

In [ ]:
# Pull the llama3.2:1b model
!ollama pull llama3.2:1b

In [32]:
import requests

TUNNEL_URL = "https://ekyrz-35-227-152-127.run.pinggy-free.link"

payload = {
    "model": "llama3.2:latest",
    "prompt": "Why is the sky blue?",
    "stream": False
}

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "User-Agent": "Mozilla/5.0"
}

try:
    response = requests.post(
        f"{TUNNEL_URL}/api/generate",
        json=payload,
        headers=headers,
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    print(data.get("response", "No response field returned"))
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
except ValueError:
    print("Response was not valid JSON")

Request failed: Expecting value: line 1 column 1 (char 0)


In [ ]:
# Install Open WebUI
!pip install open-webui

In [ ]:
# Start a pinggy tunnel for Open WebUI
import pinggy
tunnel1 = pinggy.start_tunnel(
    forwardto="localhost:8000",
)

print(f"Tunnel1 started - URLs: {tunnel1.urls}")

In [ ]:
# Start the Open WebUI server on port 8000
!open-webui serve --port 8000